# Import packages

In [1]:
import os, requests
from pprint import pprint
import pandas as pd

# Custom functions

In [2]:
from tools.create_jaspar_filters import obtain_jaspar_dataset, obtain_taxonomy_id, obtain_all_cell_lines, \
    cell_lines_df_clearing

# Download original JASPAR dataset
JASPAR: https://jaspar.elixir.no/downloads/
* Go for
    * Latest release
    * `CORE PFMs` collection for experimentally validated TF binding motifs with high quality
    * `Vertebrates` as the taxonomic group
    * `Single batch file (txt)`
    * `non-redundant` removes duplicate motifs for the same TF
    * `MEME` format can be used natively with fimo

In [3]:
jaspar_url = 'https://jaspar.elixir.no/download/data/2026/CORE/JASPAR2026_CORE_vertebrates_non-redundant_pfms_meme.txt'

jaspar_save_dir = 'input/jaspar_dataset'
os.makedirs(jaspar_save_dir, exist_ok=True)

In [4]:
file_path = os.path.join(jaspar_save_dir, jaspar_url.split('/')[-1].split('.')[0] + '_original.txt')

response = requests.get(jaspar_url)

with open(file_path, "wb") as f:
    f.write(response.content)

print("Downloaded to:", file_path)

Downloaded to: input/jaspar_dataset/JASPAR2026_CORE_vertebrates_non-redundant_pfms_meme_original.txt


# Obtain specific cell lines
Same logic can be applied to other diseases for filtering

In [5]:
filter_save_dir = 'input/filters'
os.makedirs(filter_save_dir, exist_ok=True)

## Obtain all melanoma cell lines

### (Optional) Obtain all NCIt code releated to a search term
NCI EVS API fields exploration: https://evsexplore.semantics.cancer.gov/evsexplore/evsapi

In [6]:
ncit_search_term = 'melanoma'  #Seaching "melanoma" will also return results like "cutaneous melanoma"
page_size = 500  #This is for pagerization that NCIt has. It's a protection such that not all results return in 1 call but that also means you need to iterate and pool all the returns together

In [7]:
all_concepts = obtain_jaspar_dataset(ncit_search_term, page_size)

Concepts obtained: 0 to 500
Concepts obtained: 500 to 1000
Concepts obtained: 1000 to 1500
Concepts obtained: 1500 to 1642
Total NCIt code(s) retrieved for melanoma: 1642


In [8]:
pprint(all_concepts, indent=4)

[   {   'active': True,
        'code': 'C3224',
        'conceptStatus': 'DEFAULT',
        'leaf': False,
        'name': 'Melanoma',
        'terminology': 'ncit',
        'version': '26.02d'},
    {   'active': True,
        'code': 'C91477',
        'conceptStatus': 'DEFAULT',
        'leaf': True,
        'name': 'Melanoma Pathway',
        'terminology': 'ncit',
        'version': '26.02d'},
    {   'active': True,
        'code': 'C21790',
        'conceptStatus': 'DEFAULT',
        'leaf': False,
        'name': 'Mouse Melanoma',
        'terminology': 'ncit',
        'version': '26.02d'},
    {   'active': True,
        'code': 'C103113',
        'conceptStatus': 'DEFAULT',
        'leaf': True,
        'name': 'NCI CTEP SDC Melanoma Sub-Category Terminology',
        'terminology': 'ncit',
        'version': '26.02d'},
    {   'active': True,
        'code': 'C157920',
        'conceptStatus': 'DEFAULT',
        'leaf': True,
        'name': 'Melanoma Surgery',
        'term

In [9]:
ncit_dict = {dict['code']: dict['name'] for dict in all_concepts}  #All the other fields don't matter
pprint(ncit_dict)

{'C100054': 'Conjunctival Melanocytic Intraepithelial Lesion',
 'C101364': 'KCNMA1 wt Allele',
 'C101787': 'TIL 1383I T Cell Receptor-Transduced Autologous T Cells',
 'C102753': 'Adenovirus Encoding Tyrosinase/MART-1/MAGEA6-transduced '
            'Autologous Dendritic Cell Vaccine',
 'C10303': 'CDB Regimen',
 'C103113': 'NCI CTEP SDC Melanoma Sub-Category Terminology',
 'C103830': 'Recombinant Human Hsp110-gp100 Chaperone Complex Vaccine',
 'C103862': 'Allogeneic Irradiated Melanoma Cell Vaccine CSF470',
 'C104047': 'CTAG2 wt Allele',
 'C104492': 'BAGE Gene',
 'C104493': 'BAGE wt Allele',
 'C104494': 'B Melanoma Antigen 1',
 'C104496': 'MAGEA10 Gene',
 'C104497': 'MAGEA10 wt Allele',
 'C104498': 'Melanoma-Associated Antigen 10',
 'C104499': 'MAGEA2 Gene',
 'C104500': 'MAGEA2 wt Allele',
 'C104501': 'Melanoma-Associated Antigen 2',
 'C104509': 'MAGEA1 Gene',
 'C104510': 'MAGEA1 wt Allele',
 'C104511': 'Melanoma-Associated Antigen 1',
 'C104512': 'MAGEA4 Gene',
 'C104513': 'MAGEA4 wt A

In [10]:
ncit_df = pd.DataFrame(ncit_dict.items(), columns=['code', 'concept'])
ncit_df.to_csv(
    os.path.join(filter_save_dir, f'NCIt_concepts_{ncit_search_term}.csv'),
    index=False
)  #Save NCIt outputs for future references

### Obtain taxonomy code
This can be used later to filter cell lines by species

In [11]:
species_search_term = 'Homo sapiens'

In [12]:
tax_id = obtain_taxonomy_id(species_search_term)

Taxonomy ID for Homo sapiens: 9606


### Obtain cell lines related to a search term
Cellosaurus API methods & exploration: https://api.cellosaurus.org/api-methods#/

In [13]:
cell_line_search_term = 'melanoma'
search_term_type ='di', #Because melanoma is a disease

In [14]:
all_cell_lines_data = obtain_all_cell_lines(
    cell_line_search_term,
    search_term_type,
    tax_id
)

Concepts obtained: 0 to 92
Total cell line(s) retrieved for melanoma: 92


In [15]:
cell_line_df = cell_lines_df_clearing(all_cell_lines_data)
cell_line_df = cell_line_df[[
    'cell_line_id',
    'cell_line_name',
    'organ',
    'site_type',
    'disease',
    'age',
    'sex',
    'category',
    'species'
]]  # Reorder the columns

cell_line_df

CVCL_M212 lacks derived-from-site-list
CVCL_M212 lacks derived-from-site-list
CVCL_HF01 lacks derived-from-site-list
CVCL_HF01 lacks derived-from-site-list
CVCL_HF13 lacks derived-from-site-list
CVCL_HF13 lacks derived-from-site-list
CVCL_V020 lacks derived-from-site-list
CVCL_V020 lacks derived-from-site-list
CVCL_VP94 lacks derived-from-site-list
CVCL_VP94 lacks derived-from-site-list
CVCL_F250 lacks derived-from-site-list
CVCL_F250 lacks derived-from-site-list
CVCL_B7TC lacks derived-from-site-list
CVCL_B7TC lacks derived-from-site-list
CVCL_B7TD lacks derived-from-site-list
CVCL_B7TD lacks derived-from-site-list
CVCL_C668 lacks derived-from-site-list
CVCL_C668 lacks derived-from-site-list
CVCL_C672 lacks derived-from-site-list
CVCL_C672 lacks derived-from-site-list
CVCL_C680 lacks derived-from-site-list
CVCL_C680 lacks derived-from-site-list
CVCL_A648 lacks derived-from-site-list
CVCL_A648 lacks derived-from-site-list
CVCL_1068 lacks derived-from-site-list
CVCL_1068 lacks derived-f

,cell_line_id,cell_line_name,organ,site_type,disease,age,sex,category,species
0,CVCL_M212,FRM [Human melanoma],NaN,NaN,Amelanotic melanoma,Age unspecified,Sex unspecified,Cancer cell line,Homo sapiens (Human)
1,CVCL_HF04,"Me14464,14464",Lymph node,Metastatic,Melanoma,Age unspecified,Sex unspecified,Cancer cell line,Homo sapiens (Human)
2,CVCL_HF01,Me10258,NaN,NaN,Melanoma,Age unspecified,Sex unspecified,Cancer cell line,Homo sapiens (Human)
3,CVCL_HE99,"Me4686,4686",Lymph node,Metastatic,Melanoma,Age unspecified,Sex unspecified,Cancer cell line,Homo sapiens (Human)
4,CVCL_HF02,Me1274,Lymph node,Metastatic,Melanoma,Age unspecified,Sex unspecified,Cancer cell line,Homo sapiens (Human)
...,...,...,...,...,...,...,...,...,...
87,CVCL_C680,"INT-MEL-17,Me4405,ME4405,ME-4405,ME 4405",NaN,NaN,Melanoma,83Y,Female,Cancer cell line,Homo sapiens (Human)
88,CVCL_A648,"INT-MEL-12,Me10538,ME10538,ME 10538,Me 10538,M...",NaN,NaN,Melanoma,56Y,Female,Cancer cell line,Homo sapiens (Human)
89,CVCL_1068,ACN,NaN,NaN,Melanoma,Age unspecified,Sex unspecified,Cancer cell line,Homo sapiens (Human)
90,CVCL_0V14,"Mel501A,MEL501A,501A,501 A",Not specified,Metastatic,Melanoma,Age unspecified,Male,Cancer cell line,Homo sapiens (Human)


In [16]:
cell_line_df.to_csv(f'input/filters/cell_lines_{cell_line_search_term}.csv', index=False)

### Filter melanoma cell lines

#### Melanoma brain metastasis cell lines

In [17]:
df_mbm = cell_line_df[
    cell_line_df['organ'].str.contains('brain', case=False, na=False) &
    cell_line_df['site_type'].str.contains('metastatic', case=False, na=False)
] # These are all the melanoma that metastasizes to the brain

df_mbm = df_mbm.reset_index(drop=True)
df_mbm

,cell_line_id,cell_line_name,organ,site_type,disease,age,sex,category,species


In [18]:
df_mbm.to_csv(f'input/filters/cell_lines_mbm.csv', index=False)

#### Uveal melanoma cell lines

In [19]:
df_uveal = cell_line_df[
    cell_line_df['disease'].str.contains('uvea', case=False, na=False)
] #These are all the uveal melanoma, including the in situ and metastatic ones that can go to various locations (can be further specified)

df_uveal = df_uveal.reset_index(drop=True)
df_uveal

,cell_line_id,cell_line_name,organ,site_type,disease,age,sex,category,species


In [20]:
df_mbm.to_csv(f'input/filters/cell_lines_um.csv', index=False)

## Obtain all astrocyte cell lines

### Obtain cell lines related to a search term

In [21]:
cell_line_search_term = 'astrocyte'
search_term_type = 'cell-type' #Because astrocyte is a cell type

In [22]:
all_cell_lines_data = obtain_all_cell_lines(
    cell_line_search_term,
    search_term_type,
    tax_id
)

Concepts obtained: 0 to 19
Total cell line(s) retrieved for astrocyte: 19


In [23]:
cell_line_df = cell_lines_df_clearing(all_cell_lines_data)
cell_line_df = cell_line_df[[
    'cell_line_id',
    'cell_line_name',
    'organ',
    'site_type',
    'disease',
    'age',
    'sex',
    'category',
    'species'
]]  # Reorder the columns

cell_line_df

CVCL_AQ52 lacks age
CVCL_AQ52 lacks sex
CVCL_AQ52 lacks derived-from-site-list
CVCL_AQ52 lacks derived-from-site-list
CVCL_AQ52 lacks disease-list
CVCL_4U69 lacks disease-list
CVCL_B4IH lacks disease-list
CVCL_X371 lacks disease-list
CVCL_E3G4 lacks disease-list
CVCL_E3G5 lacks disease-list
CVCL_Z595 lacks disease-list
CVCL_Z596 lacks disease-list
CVCL_5G14 lacks disease-list
CVCL_3797 lacks disease-list
CVCL_D5B6 lacks disease-list
CVCL_D5B7 lacks disease-list
CVCL_5G13 lacks disease-list
CVCL_C3HJ lacks disease-list
CVCL_VP19 lacks disease-list
CVCL_VP18 lacks disease-list
CVCL_5F78 lacks disease-list
CVCL_IM78 lacks disease-list
CVCL_B5WG lacks age
CVCL_B5WG lacks sex
CVCL_B5WG lacks disease-list


,cell_line_id,cell_line_name,organ,site_type,disease,age,sex,category,species
0,CVCL_AQ52,Astro.4U,NaN,NaN,NaN,NaN,NaN,Finite cell line,Homo sapiens (Human)
1,CVCL_4U69,"HASTR/ci35,Human ASTRocyte/conditionally immor...",Brain,In situ,NaN,4M,Sex unspecified,Conditionally immortalized cell line,Homo sapiens (Human)
2,CVCL_B4IH,A206,Fetal brain,In situ,NaN,12-16FW,Sex unspecified,Transformed cell line,Homo sapiens (Human)
3,CVCL_X371,A735,Fetal brain,In situ,NaN,12-16FW,Sex unspecified,Transformed cell line,Homo sapiens (Human)
4,CVCL_E3G4,NHA E6/E7/hTERT/Ras,Brain,In situ,NaN,Age unspecified,Sex unspecified,Transformed cell line,Homo sapiens (Human)
5,CVCL_E3G5,NHA E6/E7/hTERT/Ras/Akt,Brain,In situ,NaN,Age unspecified,Sex unspecified,Transformed cell line,Homo sapiens (Human)
6,CVCL_Z595,SK-astrocyte-IDH1,Brain,In situ,NaN,Age unspecified,Sex unspecified,Telomerase immortalized cell line,Homo sapiens (Human)
7,CVCL_Z596,SK-astrocyte-IDH1-R132H,Brain,In situ,NaN,Age unspecified,Sex unspecified,Telomerase immortalized cell line,Homo sapiens (Human)
8,CVCL_5G14,"SVG,SV40 immortalized Glial cell",Fetal brain,In situ,NaN,8-12FW,Male,Transformed cell line,Homo sapiens (Human)
9,CVCL_3797,"SVG p12,SVGp12,SVG(P12)",Fetal brain,In situ,NaN,8-12FW,Male,Transformed cell line,Homo sapiens (Human)


In [24]:
cell_line_df.to_csv(f'input/filters/cell_lines_{cell_line_search_term}.csv', index=False)